# iprPy elastic_constants_dynamic calculation

In [1]:
# Standard library imports
import datetime

# http://www.numpy.org/
import numpy as np

# https://ipython.org/
from IPython.display import display, Code, Markdown, Pretty

# https://github.com/usnistgov/atomman 
import atomman as am
import atomman.unitconvert as uc

# https://github.com/usnistgov/iprPy
import iprPy

print('Notebook last executed on', datetime.date.today(), 'using iprPy version', iprPy.__version__)

Notebook last executed on 2026-06-30 using iprPy version 0.12.a


## 1. Load calculation and view description

### 1.1. Load the calculation

In [2]:
# Load the calculation being demoed
calculation = iprPy.load_calculation('elastic_constants_dynamic')

### 1.2. Display calculation description and theory

In [3]:
# Display main docs and theory
display(Markdown(calculation.maindoc))
display(Markdown(calculation.theorydoc))

# elastic_constants_dynamic calculation style

**Lucas M. Hale**, [lucas.hale@nist.gov](mailto:lucas.hale@nist.gov?Subject=ipr-demo), *Materials Science and Engineering Division, NIST*.

## Introduction

The elastic_constants_dynamic calculation style computes the elastic constants, $C_{ij}$, for a system using the fluctuation method through computing the Born matrix.  This should provide elastic constants estimates comparable to elastic_constants_static for 0K calculations and relatively quick evaluations of elastic constants at higher temperatures.

### Version notes

- 2024-04-25 Calculation method based on the fluctuations/born matrix method finalized.
- 2026-06-25: Method updated to support the LAMMPS library interface.

### Additional dependencies

### Disclaimers

- [NIST disclaimers](http://www.nist.gov/public_affairs/disclaimer.cfm)
- This calculation does not perform any relaxations on the box dimensions.  As the elastic constants are sensitive to both temperature and pressure, be sure to properly relax your system before passing it into this calculation.
- Estimates of the second derivative of energy with respect to strain are computed numerically using a small strain value.  The computed elastic constants may be sensitive to the choice in strain.  The best values are obtained by strains that are large enough to overcome numerical issues with precision while small enough so that the elastic behavior is still in the linear regime.


## Method and Theory

This calculation method uses the [deformation–fluctuation hybrid method](https://doi.org/10.1016/j.cpc.2011.09.006) for computing elastic constants as implemented in LAMMPS under the [compute born/matrix numdiff](https://docs.lammps.org/compute_born_matrix.html) command.

With the fluctuation method, the elastic constants can be estimated by computing three terms:

$$ C_{ij} = C_{ij}^{Born} + C_{ij}^{fluc} + C_{ij}^{kin}$$

$C_{ij}^{Born}$ is the mean of the Born matrix, which is the second derivatives of the potential energy with respect to strain

$$ C_{ij}^B=\left<\frac{1}{V} \frac{\partial^2U}{\partial\epsilon_i\partial\epsilon_j} \right>$$

The LAMMPS compute born/matrix command evaluates this matrix as the simulation runs.  For the numdiff option, the calculation follows the deformation-fluctuation method and uses finite differences of the energy to approximate the derivatives.  This is done by the calculation applying linear strain fields to all atoms in the system associated with all six independent $\epsilon_{ij}$ components in positive and negative directions allowing for an estimate of the second derivative wrt to the strains. This makes this calculation available to any interatomic potential that evaluates energies but does add a dependency of the calculation on the size of the strain used.

$C_{ij}^{fluc}$ is the fluctuation (a.k.a. stress) matrix given by

$$ - \frac{V}{k_B T} \left( \left<\sigma_i \sigma_j \right> - \left<\sigma_i\right> \left<\sigma_j\right> \right), $$

where $\sigma$ is the virial stress tensor.  Note that sometimes the fluctuation term is defined without the negative included and is then subtracted from the other terms when computing $C_{ij}$.  This term is computed by regularly measuring the virial pressure of the system during the LAMMPS calculation, then computing the covariance of the values.

$C_{ij}^{kin}$ is the kinetic term, which is the "ideal gas" contribution and only depends on temperature 

 $$ C_{ij}^{kin} = \frac{N k_B T}{V} ( \delta_{ij} + (\delta_{1i} + \delta_{2i} + \delta_{3i}) * (\delta_{1j} + \delta_{2j} + \delta_{3j}) ), $$
    
where δ is the Kronecker delta. Evaluating the second part of the term, this can be simplified to
    
 $$ C_{ij}^{kin} = \frac{N k_B T}{V} \Delta_{ij}, $$
    
where $\Delta_{ij} = 2$ for $ij =11, 22, 33$, $\Delta_{ij} = 1$ for $ij = 44, 55, 66$ and $\Delta_{ij} = 0$ otherwise.    



## 2. Display the underlying code

This section displays the underlying code used when calculation.calc() is called in Section 4. It is provided here allowing any users to see and understand how the calculation works.

Feel free to modify and test the functions for yourself!  You can do this by
1. Copy the Python code displayed here into a "Code" cell and run it.
2. In Section 2.2, set "savefiles = True" and run the cell to save any supporting non-python files to the working directory.
3. In Section 4, call the primary calculation function directly rather than using calculation.calc().

### 2.1. Show code and supporting file names and content

In [4]:
# Display calculation code and supporting files
for filename, contents in calculation.files.items():
    display(Markdown(f'## Contents of file "{filename}"'))
    if filename[-3:] == '.py':
        display(Code(contents, language='python'))
    else:
        display(Pretty(contents))

## Contents of file "elastic_constants_dynamic.py"

# Python script created by Lucas Hale
import datetime
from typing import Optional, Union

# http://www.numpy.org/
import numpy as np

# https://github.com/usnistgov/atomman 
import atomman as am
import atomman.unitconvert as uc
from atomman.typing import lammpspotential, unitfloat
from atomman.lammps import LAMMPS, LAMMPSobj

def elastic_constants_dynamic(lammps_command: Union[str, LAMMPSobj],
                              system: am.System,
                              potential: lammpspotential,
                              temperature: float,
                              mpi_command: Optional[str] = None,
                              normalized_as: str = 'triclinic',
                              strainrange: float = 1e-6,
                              equilsteps: int = 20000,
                              runsteps: int = 200000,
                              thermosteps: int = 100,
                              createvelocities: bool = True,
                              randomseed: Optional[int] = None,
                              usefiles: bool = False) -> dict:
    """
    Computes elastic constants for a system during dynamic simulations using
    the LAMMPS compute born/matrix method.
    
    Parameters
    ----------
    lammps_command : str, LAMMPSEXE or LAMMPSLIB
        LAMMPS executable command, LAMMPS library name, or an atomman LAMMPS
        interface object.
    system : atomman.System
        The system to perform the calculation on.
    potential : PotentialLAMMPS or PotentialLAMMPSKIM
        The LAMMPS implemented potential to use.
    temperature : float
        The temperature to run the calculation at.
    mpi_command : str, optional
        The MPI command for running LAMMPS in parallel.  If not given, LAMMPS
        will run serially.
    normalized_as : str, optional
        This allows for the computed elastic constants matrix values to be
        normalized to the symmetries expected for a specified crystal family.
        Default value of 'triclinic' will perform no normalization.
    strainrange : float, optional
        The magnitude of strains to use to generate finite difference
        approximations for the exact virial stress.  Picking a good value may
        be dependent on the crystal structure and it is recommended to try
        multiple different values.  Default value is 1e-6.
    equilsteps : int, optional
        Number of integration steps to perform prior to performing the
        born/matrix calculation to equilibrate the system.  Default value is
        20000.
    runsteps : int, optional
        Number of integration steps to perform during the born/matrix
        calculation.  Default value is 200000.
    thermosteps : int, optional
        How often to output thermo values to sample the computed stress and
        born/matrix values.
    randomseed : int or None, optional
        A random number seed between 1 and 9000000 to use for initializing
        velocities and use with the langevin thermostat.  Default value of None
        will pick a random value.
    
    Returns
    -------
    dict
        Dictionary of results consisting of keys:
        - **'measured_pressure'** (*float*) - The mean measured pressure of the
          system.
        - **'Cij_born'** (*numpy.ndarray*) - The 6x6 tensor of the Born
          component of the Cij calculation.
        - **'Cij_fluc'** (*numpy.ndarray*) - The 6x6 tensor of the fluctuation
          component of the Cij calculation.
        - **'Cij_kin'** (*numpy.ndarray*) - The 6x6 tensor of the kinetic
          component of the Cij calculation.
        - **'C'** (*atomman.ElasticConstants*) - The total elastic constants
          normalized by the crystal symmetry specified.
    """
    logfile = 'log.lammps'
    if usefiles:
        script = 'born_matrix.in'
    else:
        script = None

    # Create a LAMMPS object if needed
    lmp = LAMMPS(lammps_command, mpi_command=mpi_command, potential=potential)

    # Conver

### 2.2. Optional: Save supporting files

Set "savefiles = True" to save files locally.  

Note that the code above should be using atomman.tools.read_calc_file() to read the files, which will read any local files with matching names if they exist or read the packaged version if the local files do not exist. This means that if you save the files locally, you can modify them and see how it affects the calculation!

In [5]:
savefiles = False

if savefiles:
    for filename, contents in calculation.files.items():
        if filename[-3:] != '.py':
            with open(filename, 'w') as f:
                f.write(contents)

## 3. Specify input parameters

### 3.1. System-specific paths

- __lammps_command__ is the LAMMPS command to use (required).
- __mpi_command__ MPI command for running LAMMPS in parallel. A value of None will run simulations serially.

In [6]:
#lammps_command = 'lmp_mpi'
lammps_command = 'F:/LAMMPS/current/bin/lmp.exe'
mpi_command = None

# Optional: check that LAMMPS works and show its version 
print(f'LAMMPS version = {am.lammps.checkversion(lammps_command)["version"]}')

LAMMPS version = 23 Jun 2022 - Update 2


### 3.2. Interatomic potential

- __potential_name__ gives the name of a potential_LAMMPS record to find and download from the iprPy library.  
- __potential__ is a potential_LAMMPS or potential_LAMMPS_KIM record object (required).

See documentation for the [potentials package](https://github.com/usnistgov/potentials/tree/master/doc) for more options on finding, loading and building potential objects (doc Notebook #s 0, 5.3, 5.4 and 7).

In [7]:
potential_name = '1999--Mishin-Y--Ni--LAMMPS--ipr1'

# Retrieve potential and parameter file(s) using atomman
potential = am.load_lammps_potential(id=potential_name, getfiles=True)

### 3.3. Initial system

- __system__ is an atomman.System to use as the starting configuration.  Here, it is taken as the final configuration from the relax_dynamic calculation.

See documentation for the [atomman package](https://github.com/lmhale99/atomman/tree/master/doc/tutorial) for more options on building and loading atomic configurations (doc Notebook #s 1.1, 1.2, 1.3, 1.4 and 1.4.*)

In [8]:
# Load final configuration from relax_dynamic
system = am.load('atom_dump', '../relax_dynamic/220000.dump', symbols='Ni')
print('# of atoms in system =', system.natoms)

# of atoms in system = 4000


### 3.4. Calculation-specific parameters

- __temperature__ is the temperature to run the calculation at.
- __normalized_as__ allows for the computed elastic constants matrix values to be normalized to the symmetries expected for a specified crystal family.  Default value of 'triclinic' will perform no normalization.
- __strainrange__ is the magnitude of strains to use to generate finite difference approximations for the exact virial stress.  Picking a good value may be dependent on the crystal structure and it is recommended to try multiple different values.  Default value is 1e-6.
- __equilsteps__ is the number of integration steps to perform prior to performing the born/matrix calculation to equilibrate the system.  Default value is 20000.
- __runsteps__ is the number of integration steps to perform during the born/matrix calculation.  Default value is 200000.
- __thermosteps__ indicates often to output thermo values to sample the computed stress and born/matrix values.
- __randomseed__ is a random number seed between 1 and 9000000 to use for initializing velocities and use with the langevin thermostat.  Default value of None will pick a random value.

In [9]:
temperature = 300.0
normalized_as = 'cubic'
strainrange = 1e-6
equilsteps = 0
runsteps = 200000
thermosteps = 100
randomseed = None

## 4. Run calculation and view results

### 4.1. Run calculation

All primary calculation method functions take a series of inputs and return a dictionary of outputs.

In [10]:
# What is calculation.calc an alias of?
calculation.calc.__module__

'iprPy.calculation.elastic_constants_dynamic.elastic_constants_dynamic'

In [11]:
results_dict = calculation.calc(lammps_command, system, potential, temperature,
                                mpi_command=mpi_command,
                                normalized_as=normalized_as,
                                strainrange=strainrange,
                                equilsteps=equilsteps,
                                runsteps=runsteps,
                                thermosteps=thermosteps,
                                randomseed=randomseed)
print(results_dict.keys())

dict_keys(['measured_pressure', 'Cij_born', 'Cij_fluc', 'Cij_kin', 'C'])


### 4.2. Report results

Values returned in the results_dict:

- **'measured_pressure'** (*float*) - The mean measured pressure of the system.
- **'Cij_born'** (*numpy.ndarray*) - The 6x6 tensor of the Born component of the Cij calculation.
- **'Cij_fluc'** (*numpy.ndarray*) - The 6x6 tensor of the fluctuation component of the Cij calculation.
- **'Cij_kin'** (*numpy.ndarray*) - The 6x6 tensor of the kinetic component of the Cij calculation.
- **'C'** (*atomman.ElasticConstants*) - The total elastic constants normalized by the crystal symmetry specified.

In [12]:
pressure_unit = 'GPa'

print(f'Mean pressure of the system ({pressure_unit}) =')
print(uc.get_in_units(results_dict['measured_pressure'], pressure_unit))

Mean pressure of the system (GPa) =
-0.13873881768117313


In [13]:
print('Born components of Cij ('+pressure_unit+') =')
for Ci in uc.get_in_units(results_dict['Cij_born'], pressure_unit):
    print('[%9.4f %9.4f %9.4f %9.4f %9.4f %9.4f]' % tuple(Ci))
print()

print('Fluctuation components of Cij ('+pressure_unit+') =')
for Ci in uc.get_in_units(results_dict['Cij_fluc'], pressure_unit):
    print('[%9.4f %9.4f %9.4f %9.4f %9.4f %9.4f]' % tuple(Ci))  
print()

print('Kinetic components of Cij ('+pressure_unit+') =')
for Ci in uc.get_in_units(results_dict['Cij_kin'], pressure_unit):
    print('[%9.4f %9.4f %9.4f %9.4f %9.4f %9.4f]' % tuple(Ci))  

Born components of Cij (GPa) =
[ 307.2280  176.4993  176.6722    0.0412   -0.0943    1.1308]
[ 176.4993  305.2502  177.1383   -0.4508    0.0128    1.4502]
[ 176.6722  177.1383  304.4282   -0.5813   -0.1186   -0.1016]
[   0.0412   -0.4508   -0.5813  152.7346    0.0159    0.0031]
[  -0.0943    0.0128   -0.1186    0.0159  152.2720   -0.0059]
[   1.1308    1.4502   -0.1016    0.0031   -0.0059  152.1017]

Fluctuation components of Cij (GPa) =
[ -12.4108   -4.3492   -4.5731    0.0498    0.0381   -0.3218]
[  -4.3492  -12.1325   -4.4982   -0.5153    0.0965   -0.0080]
[  -4.5731   -4.4982  -12.0512    0.0064    0.2083    0.2818]
[   0.0498   -0.5153    0.0064   -6.3067    0.0496    0.0829]
[   0.0381    0.0965    0.2083    0.0496   -6.2306    0.0451]
[  -0.3218   -0.0080    0.2818    0.0829    0.0451   -6.0782]

Kinetic components of Cij (GPa) =
[   0.7509    0.0000    0.0000    0.0000    0.0000    0.0000]
[   0.0000    0.7509    0.0000    0.0000    0.0000    0.0000]
[   0.0000    0.0000    0.7

The returned C ElasticConstants object is the elastic constants normalized by the specified crystal family symmetry.

In [14]:
print('Normalized Cij ('+pressure_unit+') =')
for Ci in uc.get_in_units(results_dict['C'].Cij, pressure_unit):
    print('[%9.4f %9.4f %9.4f %9.4f %9.4f %9.4f]' % tuple(Ci))

Normalized Cij (GPa) =
[ 294.1882  172.2965  172.2965    0.0000    0.0000    0.0000]
[ 172.2965  294.1882  172.2965    0.0000    0.0000    0.0000]
[ 172.2965  172.2965  294.1882    0.0000    0.0000    0.0000]
[   0.0000    0.0000    0.0000  146.5397    0.0000    0.0000]
[   0.0000    0.0000    0.0000    0.0000  146.5397    0.0000]
[   0.0000    0.0000    0.0000    0.0000    0.0000  146.5397]


To see the unnormalized Cij values, either run with normalized_as = None or 'triclinic', or simply add the three individual components.

In [15]:
Cij_raw = results_dict['Cij_born'] + results_dict['Cij_fluc'] + results_dict['Cij_kin']
print('Unnormalized Cij ('+pressure_unit+') =')
for Ci in uc.get_in_units(Cij_raw, pressure_unit):
    print('[%9.4f %9.4f %9.4f %9.4f %9.4f %9.4f]' % tuple(Ci))

Unnormalized Cij (GPa) =
[ 295.5681  172.1502  172.0992    0.0910   -0.0562    0.8090]
[ 172.1502  293.8686  172.6401   -0.9661    0.1093    1.4421]
[ 172.0992  172.6401  293.1279   -0.5749    0.0896    0.1801]
[   0.0910   -0.9661   -0.5749  146.8033    0.0655    0.0860]
[  -0.0562    0.1093    0.0896    0.0655  146.4168    0.0392]
[   0.8090    1.4421    0.1801    0.0860    0.0392  146.3989]


### 4.3. Optional: Clean calculation files

The calculation may generate output files when it runs.  Calling calculation.clean_files() will delete any generated files to keep the workspace clean.

In [16]:
calculation.clean_files()